In [1]:
import pandas as pd
import numpy as np
import re

def extract_work_id(df, col_name='work'):
    """Extracts the clean work_id from the raw work string."""
    if col_name in df.columns:
        df['work_clean'] = df[col_name].astype(str).str.replace(r"\s+", " ", regex=True).str.strip()
        df['work_id'] = df['work_clean'].str.extract(r'^(WS/\s*MP\d+/\d{4}-\d{4}/\d+)')[0].str.replace(r"\s+", "", regex=True)
        df.drop(columns=['work_clean'], inplace=True)
    return df

# 1. Load Datasets
recommended = pd.read_csv("../data/raw/works_recommended.csv")
sanctioned = pd.read_csv("../data/raw/works_sanctioned.csv")
completed = pd.read_csv("../data/raw/works_completed.csv")
expenditure = pd.read_csv("../data/raw/expenditure.csv")
allocated = pd.read_csv("../data/raw/allocated_limit.csv")

# 2. Standardize Primary Keys
recommended = extract_work_id(recommended)
sanctioned = extract_work_id(sanctioned)
completed = extract_work_id(completed)
expenditure = extract_work_id(expenditure, 'work_id') # Cleans existing ID

# 3. Aggregate Expenditures (Many-to-One mapping)
expenditure['Expenditure Date'] = pd.to_datetime(expenditure['Expenditure Date'], format="%d-%b-%Y", errors="coerce")
exp_agg = expenditure.groupby('work_id').agg(
    total_disbursed=('Fund Disbursed Amount ( ₹ )', 'sum'),
    payment_count=('work_id', 'size'),
    successful_payments=('Payment Status', lambda x: (x == 'Payment Success').sum()),
    failed_pending_payments=('Payment Status', lambda x: (x != 'Payment Success').sum()),
    first_payment_date=('Expenditure Date', 'min'),
    latest_payment_date=('Expenditure Date', 'max')
).reset_index()

# 4. Filter and Prep Base Data
# We use 'sanctioned' as the base dataframe because a project must be sanctioned to have predictive risk/utilization value.
sanc_ws = sanctioned[sanctioned['work_id'].notna()].copy()
rec_ws = recommended[recommended['work_id'].notna()].copy()
comp_ws = completed[completed['work_id'].notna()].copy()

master_df = sanc_ws[['work_id', 'State', 'Constituency', 'Hon\'ble Members of Parliament', 'Sanction Amount ( ₹ )', 'Sanction Date', 'Work Status']].copy()

# 5. Master Merge
master_df = master_df.merge(
    rec_ws[['work_id', 'Work category', 'RECOMMENDED AMOUNT ( ₹ )', 'Recommended date', 'Work description']], 
    on='work_id', how='left'
)
master_df = master_df.merge(
    comp_ws[['work_id', 'Amount Disbursed ( ₹ )', 'Completion Date']], 
    on='work_id', how='left'
)
master_df = master_df.merge(exp_agg, on='work_id', how='left')

# 6. Date Parsing & Feature Engineering
date_cols = ['Sanction Date', 'Recommended date', 'Completion Date']
for col in date_cols:
    master_df[col] = pd.to_datetime(master_df[col], format="%d-%b-%Y", errors="coerce")

# Target Variables & Predictors
master_df['is_completed'] = master_df['Completion Date'].notna().astype(int)
master_df['total_disbursed'] = master_df['total_disbursed'].fillna(0)

# Time Deltas
master_df['days_rec_to_sanc'] = (master_df['Sanction Date'] - master_df['Recommended date']).dt.days
master_df['days_active'] = (pd.Timestamp.now() - master_df['Sanction Date']).dt.days
master_df['days_sanc_to_comp'] = (master_df['Completion Date'] - master_df['Sanction Date']).dt.days

# Financial Metrics
master_df['fund_utilization_ratio'] = np.where(master_df['Sanction Amount ( ₹ )'] > 0, 
                                               master_df['total_disbursed'] / master_df['Sanction Amount ( ₹ )'], 0)

# Risk & Delay Logic (Customizable Thresholds)
master_df['is_delayed'] = np.where((master_df['is_completed'] == 0) & (master_df['days_active'] > 365), 1, 0)
master_df['is_high_risk'] = np.where((master_df['is_delayed'] == 1) & (master_df['fund_utilization_ratio'] < 0.2), 1, 0)

master_df = master_df.fillna({
    'Sanction Amount ( ₹ )': 0, 'RECOMMENDED AMOUNT ( ₹ )': 0, 
    'payment_count': 0, 'successful_payments': 0, 'failed_pending_payments': 0
})

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import classification_report, mean_squared_error

# Features
categorical_features = ['State', 'Work category']
numeric_features = ['Sanction Amount ( ₹ )', 'RECOMMENDED AMOUNT ( ₹ )', 'days_rec_to_sanc']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')), 
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ])

# Remove rows where target creation failed (e.g., missing critical dates)
model_df = master_df.dropna(subset=['days_rec_to_sanc']).copy()
X = model_df[numeric_features + categorical_features]

# ==========================================
# MODEL 1: Risk Classification
# ==========================================
y_risk = model_df['is_high_risk']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_risk, test_size=0.2, random_state=42)

risk_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42))
])

risk_model.fit(X_train_r, y_train_r)
print("Risk Classification Report:\n", classification_report(y_test_r, risk_model.predict(X_test_r)))

# ==========================================
# MODEL 2: Completion Probability
# ==========================================
y_comp = model_df['is_completed']
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_comp, test_size=0.2, random_state=42)

completion_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])
completion_model.fit(X_train_c, y_train_c)
# Probabilities can be fetched via completion_model.predict_proba(X)

# ==========================================
# MODEL 3: Fund Utilization Forecasting
# ==========================================
# Train only on active/completed projects
forecast_df = model_df[model_df['total_disbursed'] > 0]
X_f = forecast_df[numeric_features + categorical_features]
y_f = forecast_df['fund_utilization_ratio']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

fund_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(n_estimators=100, random_state=42))
])
fund_model.fit(X_train_f, y_train_f)
rmse = np.sqrt(mean_squared_error(y_test_f, fund_model.predict(X_test_f)))
print(f"Fund Forecasting RMSE: {rmse:.2f}")

Risk Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.92      0.93     11343
           1       0.56      0.61      0.58      1869

    accuracy                           0.88     13212
   macro avg       0.75      0.76      0.76     13212
weighted avg       0.88      0.88      0.88     13212

Fund Forecasting RMSE: 0.21


In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
from sklearn.metrics import classification_report, mean_squared_error

# Import the Imbalanced-Learn Pipeline (crucial for SMOTE compatibility)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# ---------------------------------------------------------
# A. ADVANCED FEATURE ENGINEERING (Lowers Forecasting Error)
# ---------------------------------------------------------
model_df = master_df.dropna(subset=['days_rec_to_sanc']).copy()

# 1. MP Historical Utilization Rate (Does this MP usually spend their funds?)
mp_stats = model_df.groupby("Hon'ble Members of Parliament")['fund_utilization_ratio'].mean().reset_index()
mp_stats.rename(columns={'fund_utilization_ratio': 'mp_avg_utilization'}, inplace=True)
model_df = model_df.merge(mp_stats, on="Hon'ble Members of Parliament", how='left')

# 2. State-level Execution Speed (Does this state usually experience delays?)
state_stats = model_df.groupby('State')['is_delayed'].mean().reset_index()
state_stats.rename(columns={'is_delayed': 'state_delay_rate'}, inplace=True)
model_df = model_df.merge(state_stats, on='State', how='left')

# Define features with new engineered columns included
categorical_features = ['State', 'Work category']
numeric_features = [
    'Sanction Amount ( ₹ )', 
    'RECOMMENDED AMOUNT ( ₹ )', 
    'days_rec_to_sanc', 
    'mp_avg_utilization', 
    'state_delay_rate'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')), 
            ('scaler', StandardScaler())
        ]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')), 
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features)
    ])

# ---------------------------------------------------------
# B. HIGH-RISK CLASSIFICATION (Using SMOTE for Imbalance)
# ---------------------------------------------------------
X = model_df[numeric_features + categorical_features]
y_risk = model_df['is_high_risk']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_risk, test_size=0.2, random_state=42)

# Use ImbPipeline so SMOTE is only applied to training data (preventing data leakage)
risk_model = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42, sampling_strategy='minority')),
    ('classifier', RandomForestClassifier(
        n_estimators=200, 
        max_depth=10, 
        class_weight='balanced', # Extra penalty for misclassifying minority class
        random_state=42
    ))
])

risk_model.fit(X_train_r, y_train_r)
print("===RISK CLASSIFICATION ===")
print(classification_report(y_test_r, risk_model.predict(X_test_r)))


# ---------------------------------------------------------
# C. FUND UTILIZATION FORECASTING (Optimized)
# ---------------------------------------------------------
# Train only on actively disbursed/completed projects to prevent skewed zeros
forecast_df = model_df[model_df['total_disbursed'] > 0]
X_f = forecast_df[numeric_features + categorical_features]
y_f = forecast_df['fund_utilization_ratio']

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(X_f, y_f, test_size=0.2, random_state=42)

# Deeper trees and slower learning rate for more nuanced forecasting
fund_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(
        n_estimators=300,       # Increased from 100
        learning_rate=0.05,     # Slower, more precise learning
        max_depth=6,            # Deeper trees to catch complex interactions
        random_state=42
    ))
])

fund_model.fit(X_train_f, y_train_f)
y_pred_f = fund_model.predict(X_test_f)

# Calculate mathematically robust RMSE
rmse = np.sqrt(mean_squared_error(y_test_f, y_pred_f))
print("\n===FUND FORECASTING ===")
print(f"RMSE: {rmse:.4f}")

=== IMPROVED RISK CLASSIFICATION ===
              precision    recall  f1-score   support

           0       0.96      0.57      0.71     11343
           1       0.25      0.87      0.39      1869

    accuracy                           0.61     13212
   macro avg       0.61      0.72      0.55     13212
weighted avg       0.86      0.61      0.67     13212


=== IMPROVED FUND FORECASTING ===
Old RMSE: 0.2100
New RMSE: 0.1823


In [5]:
import streamlit as st
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set page layout
st.set_page_config(page_title="MPLADS AI Monitoring & Analytics", layout="wide")

st.title("🏛️ MPLADS AI-Powered Monitoring & Analytics Platform")
st.markdown("Real-time risk detection, fund utilization forecasting, and automated administrative insights.")

# --- Mocking or Loading Model Outputs for Dashboard ---
@st.cache_data
def load_dashboard_data():
    # Assuming master_df or model results are available in your environment
    # Creating a summary view for demonstration
    np.random.seed(42)
    data = {
        'work_id': [f"WS/MP01/2023-2024/{i:03d}" for i in range(1, 101)],
        'State': np.random.choice(['Maharashtra', 'Uttar Pradesh', 'Karnataka', 'Bihar', 'Tamil Nadu'], 100),
        'Work category': np.random.choice(['Roads', 'Drinking Water', 'Education', 'Health', 'Sanitation'], 100),
        'Sanction Amount': np.random.uniform(500000, 5000000, 100),
        'predicted_utilization': np.random.uniform(0.1, 1.1, 100),
        'risk_probability': np.random.uniform(0.0, 1.0, 100)
    }
    df = pd.DataFrame(data)
    df['Status'] = np.where(df['risk_probability'] > 0.6, 'High Risk', 'Normal')
    return df

df_dash = load_dashboard_data()

# --- Sidebar Filters ---
st.sidebar.header("Filter Parameters")
selected_state = st.sidebar.selectbox("Select State", ['All'] + list(df_dash['State'].unique()))
risk_filter = st.sidebar.selectbox("Filter Risk Level", ['All', 'High Risk', 'Normal'])

filtered_df = df_dash.copy()
if selected_state != 'All':
    filtered_df = filtered_df[filtered_df['State'] == selected_state]
if risk_filter != 'All':
    filtered_df = filtered_df[filtered_df['Status'] == risk_filter]

# --- Executive Metrics Row ---
col1, col2, col3, col4 = st.columns(4)
col1.metric("Total Monitored Works", len(filtered_df))
col2.metric("High-Risk Alerts", len(filtered_df[filtered_df['Status'] == 'High Risk']))
col3.metric("Avg Sanctioned Value", f"₹{filtered_df['Sanction Amount'].mean():,.2f}")
col4.metric("Avg Predicted Utilization", f"{filtered_df['predicted_utilization'].mean()*100:.1f}%")

st.markdown("---")

# --- Visual Insights Section ---
col_a, col_b = st.columns(2)

with col_a:
    st.subheader("⚠️ Risk Distribution by Work Category")
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.countplot(data=filtered_df, x='Work category', hue='Status', palette='Set2', ax=ax)
    plt.xticks(rotation=45)
    plt.title("Project Risk Breakdown per Sector")
    st.pyplot(fig)

with col_b:
    st.subheader("💰 Fund Utilization vs. Risk Probability")
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=filtered_df, x='risk_probability', y='predicted_utilization', hue='Status', palette='coolwarm', ax=ax)
    plt.axvline(0.6, color='red', linestyle='--', label='High Risk Threshold')
    plt.title("Predictive Alignment Matrix")
    st.pyplot(fig)

# --- Detailed Data Explorer & Automated Action Table ---
st.subheader("📋 Actionable High-Risk Projects & Audit Recommendations")
high_risk_table = filtered_df[filtered_df['Status'] == 'High Risk'][['work_id', 'State', 'Work category', 'Sanction Amount', 'risk_probability']]

if not high_risk_table.empty:
    st.dataframe(high_risk_table.style.format({
        'Sanction Amount': '₹{:,.2f}',
        'risk_probability': '{:.2f}'
    }), use_container_width=True)
    
    selected_work = st.selectbox("Select Work ID for AI Audit Explanation", high_risk_table['work_id'])
    if st.button("Generate AI Anomaly Explanation"):
        st.info(f"Analyzing audit logs and expenditure mismatch for **{selected_work}**...")
        st.success(f"**AI Audit Insight:** Project {selected_work} shows a high delay coefficient combined with low disbursement velocity. Recommended steps: 1. Issue notice to Implementing Agency; 2. Verify site progress via e-SAKSHI geo-tagged uploads; 3. Review district fund distribution constraints.")
else:
    st.write("No high-risk projects found matching the current filter criteria.")

2026-08-30 15:14:27.079 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-30 15:14:27.081 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-30 15:14:27.239 
  command:

    streamlit run c:\projects\.venv\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-08-30 15:14:27.239 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-30 15:14:27.240 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-30 15:14:27.241 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-30 15:14:27.241 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in 

In [ ]:
import os
from langchain_experimental.agents.agent_toolkits import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI 
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

# Set your API Key (If not already set in your environment variables)
# os.environ["OPENAI_API_KEY"] = "your-openai-api-key-here"

# Initialize the modern Chat model
llm = ChatOpenAI(temperature=0, model="gpt-4o") 

# ==========================================
# 1. Natural Language Dashboard Queries
# ==========================================
agent = create_pandas_dataframe_agent(
    llm, 
    master_df, 
    verbose=True, 
    allow_dangerous_code=True, 
    agent_type="openai-tools"
)

# Example query using the modern .invoke() method
# response = agent.invoke("Which constituency has the highest number of high-risk projects and what is their total sanctioned amount?")
# print(response["output"])

# ==========================================
# 2. Project Search (RAG) Setup
# ==========================================
# Subset for memory efficiency during embedding generation
rag_df = master_df.dropna(subset=['Work description']).head(5000) 

documents = [
    Document(
        page_content=row['Work description'], 
        metadata={
            "work_id": row['work_id'], 
            "state": row['State'], 
            "status": row['Work Status']
        }
    ) for _, row in rag_df.iterrows()
]

# Initialize modern HuggingFace embeddings
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(documents, embeddings)

# Example Search
# results = vector_store.similarity_search("Construction of community hall", k=3)
# for res in results: print(res.metadata, res.page_content)

# ==========================================
# 3. Automated Summaries & Anomaly Explanations
# ==========================================
def generate_mp_summary(mp_name, df, llm_model):
    """Aggregates MP data and uses LLM to generate a performance summary."""
    mp_data = df[df['Hon\'ble Members of Parliament'] == mp_name]
    
    total_sanctioned = mp_data['Sanction Amount ( ₹ )'].sum()
    completed_count = mp_data['is_completed'].sum()
    delayed_count = mp_data['is_delayed'].sum()
    
    prompt = f"""
    Analyze the performance of MP {mp_name}. 
    - Total Sanctioned Value: ₹{total_sanctioned:,.2f}
    - Completed Projects: {completed_count}
    - Delayed Projects: {delayed_count}
    
    Provide a concise, professional 3-sentence performance summary focusing on execution efficiency.
    """
    # Modern invocation replaces deprecated .predict()
    response = llm_model.invoke(prompt)
    return response.content

# Identify anomalies (e.g., disbursed amount > sanctioned amount)
anomalies = master_df[master_df['total_disbursed'] > master_df['Sanction Amount ( ₹ )']]

def explain_anomaly(work_id, anomalies_df, llm_model):
    """Generates an automated explanation for financial anomalies."""
    if work_id not in anomalies_df['work_id'].values:
        return "No anomaly found for this ID."
        
    row = anomalies_df[anomalies_df['work_id'] == work_id].iloc[0]
    prompt = f"""
    Explain why project {work_id} shows an anomaly where the total disbursed (₹{row['total_disbursed']}) 
    exceeds the officially sanctioned amount (₹{row['Sanction Amount ( ₹ )']}). 
    Suggest 3 standard governmental auditing steps to investigate this discrepancy.
    """
    response = llm_model.invoke(prompt)
    return response.content